**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Recurrent Neural Networks

The [Kalman workshop](./Intro_AdFilt_KF.ipynb) tracked a hidden state with a *hand-written* linear model. An RNN keeps the same architecture of ideas — hidden state in, observation out, state carried forward — but **learns the dynamics from data**, nonlinearity included. We build one in PyTorch and race it against the classical baselines on a noisy oscillator.

## 0. Introduction

Feedforward networks ([ANN workshop](../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb)) see each input in isolation. Sequences need *memory*: the RNN adds a loop —

$$\mathbf{h}_t = \tanh(W_{xh}\, \mathbf{x}_t + W_{hh}\, \mathbf{h}_{t-1} + \mathbf{b}) \qquad \mathbf{y}_t = W_{hy}\, \mathbf{h}_t$$

Compare the Kalman filter's $\hat{\mathbf{x}}_k = F\hat{\mathbf{x}}_{k-1} + K_k(\cdot)$: same skeleton, but $F$, $K$ are now *learned* and wrapped in a nonlinearity.

## 1. Pre-requisites

- [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) — tensors, `nn.Module`, the training loop.
- [Adaptive Filtering: Kalman](./Intro_AdFilt_KF.ipynb) — the state-space worldview.
- Install: `pip install torch` (CPU is fine for this workshop).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)
print(torch.__version__)

2.13.0+cpu


---
### 🕐 Session 1 of 2 — *RNNs as Learned State-Space Models* (~35 min)
**Goal:** understand recurrence, backprop through time, vanishing gradients, and the LSTM fix.
**Builds on:** [Kalman](./Intro_AdFilt_KF.ipynb); [ANN](../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb). &nbsp; **Feeds into:** Session 2 (training on a real sequence).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: RNNs as Learned State-Space Models</b></summary>

**Timing (~35 min).** 8 min the Kalman↔RNN correspondence · 8 min unrolling and BPTT · 10 min vanishing gradients with the demo · 9 min the LSTM fix.

**Board first — write the two recursions above each other.** Kalman: $\hat x_k = F\hat x_{k-1} + K_k(\cdot)$. RNN: $h_t = \tanh(W_{hh}h_{t-1} + W_{xh}x_t + b)$. Same skeleton — state in, state out, observation folded in. Two differences, and both matter: the RNN *learns* its $F$ and $K$ from data instead of being handed them, and it wraps the update in a nonlinearity. Ask which you would prefer when you know the physics (Kalman, easily — it is optimal and needs no data) and when you do not (the RNN). This framing costs two minutes and makes the whole workshop feel like a continuation rather than a new subject.

**The single sentence that makes BPTT click.** "An RNN run for 50 steps is a 50-layer feedforward network whose layers share one weight matrix." Everything follows: backprop is ordinary backprop on the unrolled graph, and the only novelty is that a shared weight collects gradient contributions from every step it participated in. Students who have done the [ANN workshop](../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) then have nothing new to learn mechanically.

**Connect vanishing gradients to two things they already know.** Gradient flowing back $T$ steps is multiplied by roughly the same gain $T$ times, so it behaves like $g^T$ — the geometric series dichotomy from [Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb), and the unit-circle stability boundary from [Filter Design](../Intro_DSP/Filter_Design.ipynb). A pole inside the circle means a decaying impulse response *and* a vanishing gradient; they are the same statement. This room can see that; a generic ML audience cannot, so use the advantage.

**Ask the room.** "Gain 0.9, 60 steps back — what factor survives?" Let them compute $0.9^{60} \approx 0.0018$. Then 1.1: $1.1^{60} \approx 304$. There is no safe setting, only a knife-edge, and that is precisely why the architecture had to change rather than the initialization.

**Why the LSTM actually works — one point, made precisely.** The cell state updates by **addition**: $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$. Gradient along that path is multiplied by the forget gate rather than by a weight matrix, and with $f_t \approx 1$ it passes through essentially unattenuated. Contrast with the plain RNN, where every step multiplies by $W_{hh}$. It is a highway with learned on-ramps, and the additive structure is the whole trick — the same insight that later shows up as residual connections in [transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb).

**Misconception.** "LSTMs solve the vanishing gradient problem." They *mitigate* it, buying maybe hundreds of steps rather than tens. They do not eliminate it — which is exactly why the [SSM workshop](./State_Space_Models_Kalman_to_Mamba.ipynb) can beat an LSTM outright at 400-step recall. Flag this if that workshop is on the syllabus.

**Note the demo is deliberately linear.** `W = gain * torch.eye(8)` with no $\tanh$, so the effect is visible in its purest form. Real RNNs have saturating nonlinearities that complicate the picture; say so, or a sharp student will object.
</details>

## 2. Theory

### 2.1. Unrolling & Backprop Through Time

💡 **Intuition.** To train an RNN, *unroll* the loop: an RNN run for 50 steps is a 50-layer feedforward network **whose layers all share the same weights**. Backprop works as usual on the unrolled graph ("backprop through time"); the only twist is that each weight receives blame from *every* time step it participated in.

### 2.2. Vanishing & Exploding Gradients

💡 **Intuition.** Blame flowing back through $T$ steps gets multiplied by (roughly) $W_{hh}$'s gain $T$ times. Gain a hair below 1 ⇒ the signal *vanishes* exponentially — the network can't learn long dependencies. A hair above 1 ⇒ it *explodes*. This is the same geometric-series dichotomy as $\sum x^n$ in [Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb), and the same stability boundary as an IIR pole crossing the unit circle in [Filter Design](../Intro_DSP/Filter_Design.ipynb).

In [2]:
# Watch it happen: norm of d(h_T)/d(h_0) through a toy linear recurrence h_t = W h_{t-1}
for gain, label in [(0.9, "gain 0.9 → vanishes"), (1.1, "gain 1.1 → explodes")]:
    W = gain * torch.eye(8)
    g = torch.eye(8)
    norms = []
    for t in range(60):
        g = W.T @ g
        norms.append(g.norm().item())
    plt.semilogy(norms, label=label)
plt.legend(); plt.grid(True)
plt.xlabel("steps back in time"); plt.ylabel("gradient norm (log)")
plt.title("Why plain RNNs forget (or blow up)")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1845167/3553919686.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two straight lines on a log axis — which is the point, because straight on a log scale means *exponential*. At gain 0.9 the gradient norm falls to about $0.9^{60} \approx 1.8\times10^{-3}$ of its starting value after 60 steps back; at gain 1.1 it climbs to about $1.1^{60} \approx 3\times10^{2}$.

The consequences are asymmetric, and both are bad. Vanishing means the weights simply receive no instruction about events 60 steps ago — training does not fail loudly, it silently converges to a model that uses only recent history, and you would never know from the loss curve that a long-range dependency was ignored. Exploding is the louder failure: one oversized update destroys the weights, which is why the training loop below carries `clip_grad_norm_` as a seatbelt.

**The knife-edge is the real message.** These are not two settings to avoid between which a safe region lies. Any gain below 1 vanishes and any gain above 1 explodes, both exponentially; only gain exactly 1 is neutral, and gradient descent has no reason to sit there. You cannot initialize your way out — which is why the fix had to be architectural.

Two connections this room already owns. The behaviour is the geometric series $\sum g^n$ from [Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb), converging or diverging on whether $|g| < 1$. And it is the unit-circle stability boundary from [Filter Design](../Intro_DSP/Filter_Design.ipynb): a pole inside the circle gives a decaying impulse response, which is the *same fact* as a vanishing gradient viewed from the other end of the network. "This RNN forgets" and "this filter's impulse response has decayed" are one statement.

**And the LSTM's answer, in one line.** Route memory through a cell state updated by **addition** — $c_t = f_t\odot c_{t-1} + i_t \odot \tilde c_t$ — so gradient along that path is multiplied by the forget gate rather than by a weight matrix. With $f_t$ near 1 it passes almost unattenuated. Note this only mitigates the problem: LSTMs reliably handle hundreds of steps rather than tens, not thousands, which is precisely the gap the [state-space workshop](./State_Space_Models_Kalman_to_Mamba.ipynb) exploits.

### 2.3. LSTM: a Gated Memory Cell

The LSTM routes memory through an additive *cell state* $\mathbf{c}_t$ guarded by three learned gates — forget ($f$), input ($i$), output ($o$):

$$\mathbf{c}_t = f_t \odot \mathbf{c}_{t-1} + i_t \odot \tilde{\mathbf{c}}_t$$

Because $\mathbf{c}_t$ is updated by **addition** rather than repeated matrix multiplication, gradients can flow back along it without the exponential gain — a highway with learned on/off-ramps. (GRUs are the two-gate budget version; same spirit.)

---
### 🕐 Session 2 of 2 — *Sequence Prediction in Practice* (~40 min)
**Goal:** train an LSTM to forecast a noisy nonlinear oscillation; compare against classical baselines.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Sequence Prediction in Practice</b></summary>

**Timing (~40 min).** 5 min the signal and the windowing framing · 8 min the model · 10 min training (runs in about a minute — talk over it) · 15 min the bake-off, which is the session · 2 min buffer.

**The bake-off is the point of this session — protect its time.** Everything before it is setup. If you are running late, cut the architecture discussion, not the comparison. Students arrive expecting "we trained an LSTM and it worked"; they should leave having watched a linear model beat it, and understanding why that was predictable.

**Set the trap deliberately.** Do not signal the result in advance. Ask the room to predict the ranking before running the comparison cell — almost everyone puts the LSTM first. Then reveal: linear AR 0.01421, LSTM 0.01776, persistence 0.03177. The surprise is the pedagogy, so let them commit first.

**Then supply the number that explains it.** The signal is `sin(phase) + 0.1*randn`, so the noise variance is $0.1^2 = 0.01$ and *no* predictor can beat a one-step MSE of 0.01. That reframes everything: AR sits at 1.42× the floor and the LSTM at 1.78×, so both are close to optimal and there was almost nothing left to win. The LSTM did not fail — it succeeded at a task that was already nearly solved, using 4513 parameters where 40 sufficed. Students find "the ceiling was 0.01" far more convincing than "linear models are underrated."

**Misconception.** "The LSTM lost because it was undertrained." Partly, but the deeper reason is structural: one-step-ahead prediction of a smooth oscillation is a nearly linear problem, and a linear model is exactly the right hypothesis class. More epochs would close the gap toward AR's score and could not go meaningfully past it, because the floor is there. Capacity is not the binding constraint.

**Run the flip live if you have ten minutes.** Change the horizon to 20 steps ahead, or raise the frequency wobble to `0.8 → 3.0`, and re-run. The ranking inverts, because iterated multi-step prediction is where nonlinearity actually pays. Watching the same code produce the opposite conclusion is the most valuable thing that can happen in this session — it shows the lesson is "match the model to the task," not "linear models always win."

**Ask the room.** "Why is persistence in this table at all?" Because it is free, it has no parameters, and on smooth signals it is embarrassingly hard to beat — here it is only 3.2× the noise floor with no fitting whatsoever. A forecaster that cannot beat persistence has demonstrated nothing, and a paper without that baseline has not been evaluated. This is the professional habit the session is trying to install.

**Point at `clip_grad_norm_`.** It is Session 1's exploding-gradient theory appearing as one line of production code. Students rarely connect the theory cell to the seatbelt; name it explicitly.
</details>

## 3. Application: Forecasting a Noisy Oscillator

The target: a frequency-wobbling sinusoid — nonlinear enough that a fixed linear model struggles, structured enough to be learnable.

In [3]:
def make_signal(T=3000):
    t = np.arange(T) * 0.01
    inst_freq = 2.0 + 0.8 * np.sin(2 * np.pi * 0.05 * t)      # slowly wobbling frequency
    phase = 2 * np.pi * np.cumsum(inst_freq) * 0.01
    return (np.sin(phase) + 0.1 * rng.standard_normal(T)).astype(np.float32)

sig = make_signal()
split = 2400
train_sig, test_sig = sig[:split], sig[split:]

plt.figure(figsize=(9, 2.2))
plt.plot(sig[:800])
plt.title("The oscillator: frequency drifts, noise everywhere")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1845167/148680797.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** A sinusoid whose instantaneous frequency itself oscillates slowly, buried in noise. Two properties make it the right test signal: the frequency drift means no *single* fixed sinusoid describes it, so there is genuine structure to learn; and the additive `0.1 * randn` sets a hard limit on how well anyone can do. Hold onto that noise term — its variance, 0.01, is the floor every model in this workshop will be measured against.

### 3.1. Windowing the Sequence

Supervised framing: given the last `L` samples, predict the next one. (The same tapped-delay-line trick as the adaptive filters in the [APA workshop](./Intro_AdFilt_APA.ipynb) — the input representation is identical, only the model changes.)

In [4]:
L = 40

def windows(x, L):
    X = np.lib.stride_tricks.sliding_window_view(x, L)[:-1]    # (N, L)
    y = x[L:]                                                  # next sample
    return torch.from_numpy(X.copy()).unsqueeze(-1), torch.from_numpy(y.copy())

Xtr, ytr = windows(train_sig, L)
Xte, yte = windows(test_sig, L)
print("train windows:", tuple(Xtr.shape), " test windows:", tuple(Xte.shape))

train windows: (2360, 40, 1)  test windows: (560, 40, 1)


**What just happened.** 2360 training windows of shape (40, 1), 560 for test. `sliding_window_view` produces these as a *view* rather than a copy, which is why the `.copy()` is needed before handing them to torch — overlapping windows otherwise share memory, and in-place operations would corrupt neighbours.

Worth noting what this framing costs. Turning a sequence into independent (window, next-sample) pairs discards the fact that consecutive windows overlap in 39 of their 40 samples, so the "independent samples" assumption behind shuffling and batching is a convenient fiction. It works well in practice here, and it is also why the train/test split must be *chronological* — shuffling before splitting would leak nearly-identical windows across the boundary and produce a test score that means nothing.

In [5]:
class Forecaster(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):                 # x: (batch, L, 1)
        out, _ = self.lstm(x)             # out: (batch, L, hidden)
        return self.head(out[:, -1]).squeeze(-1)   # predict from the LAST hidden state

model = Forecaster()
print(sum(p.numel() for p in model.parameters()), "parameters")

4513 parameters


**What just happened.** 4513 parameters. Keep that number next to the linear AR baseline coming up, which will use **40** — a 113× difference in model size, on a task where the linear model is about to win.

Note also `self.head(out[:, -1])`: the prediction comes from the *last* hidden state only. All 40 timesteps are processed, but everything relevant must survive compressed into one 32-dimensional vector at the end. That bottleneck is the defining feature of recurrent architectures, and it is precisely what attention removes by letting the output read every position directly.

In [6]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)

model.train()
for epoch in range(8):
    total = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # the exploding-gradient seatbelt
        opt.step()
        total += loss.item() * len(xb)
    print(f"epoch {epoch}: train MSE {total / len(Xtr):.5f}")

epoch 0: train MSE 0.42116
epoch 1: train MSE 0.28398


epoch 2: train MSE 0.16730
epoch 3: train MSE 0.06473
epoch 4: train MSE 0.02403


epoch 5: train MSE 0.02086
epoch 6: train MSE 0.01896


epoch 7: train MSE 0.01820


**What just happened.** Train MSE falls from 0.42 to 0.018 over eight epochs, with most of the drop between epochs 2 and 5 and a clear flattening by epoch 7. The model is converging, and the curve is well-behaved — no oscillation, no divergence.

Two details in the loop are worth naming. `clip_grad_norm_(..., 1.0)` is Session 1's exploding-gradient theory as a single line of production code: it rescales any gradient whose norm exceeds 1, so a single bad batch cannot destroy the weights. It is cheap insurance and essentially standard in recurrent training. And the model predicts from `out[:, -1]` — the last hidden state only — so the LSTM must compress everything relevant from 40 timesteps into one 32-dimensional vector. That bottleneck is the recurrent design in miniature, and it is exactly what attention removes by letting every position be read directly.

**Read the flattening correctly.** The curve levelling near 0.018 does not mean the model has stopped improving because it has hit its capacity. It is approaching the *noise floor*: the signal carries additive noise with variance $0.1^2 = 0.01$, and no model can predict noise, so a training MSE of 0.018 is already within a factor of two of the best achievable value. Loss curves flatten for very different reasons — exhausted capacity, a bad learning rate, or an irreducible floor — and distinguishing them is what tells you whether a bigger model would help. Here it would not.

Keep that 0.01 in mind for the next cell. Without it, the bake-off numbers are three decimals with no scale; with it, they are three models measured against a known optimum.

### 3.2. The Bake-Off

Baselines every forecasting paper should be forced to include:

1. **Persistence** — "tomorrow = today." Embarrassingly strong on smooth signals.
2. **Linear AR(L)** — least-squares fit on the same windows: exactly the *Wiener* solution from the [APA workshop](./Intro_AdFilt_APA.ipynb), fitted in batch.

In [7]:
model.eval()
with torch.no_grad():
    pred_lstm = model(Xte).numpy()

pred_persist = Xte[:, -1, 0].numpy()

# Linear AR via least squares on the training windows
A = Xtr.squeeze(-1).numpy(); b = ytr.numpy()
w_ar, *_ = np.linalg.lstsq(A, b, rcond=None)
pred_ar = Xte.squeeze(-1).numpy() @ w_ar

truth = yte.numpy()
for name, pred in [("persistence", pred_persist), ("linear AR", pred_ar), ("LSTM", pred_lstm)]:
    print(f"{name:12s} test MSE: {np.mean((pred - truth)**2):.5f}")

persistence  test MSE: 0.03177
linear AR    test MSE: 0.01421
LSTM         test MSE: 0.01776


**What just happened.** The ranking is persistence 0.03177, **linear AR 0.01421**, LSTM 0.01776. Forty least-squares coefficients beat 4513 trained parameters, and the cell below is right that this is the most important lesson in the workshop. But the table becomes far more informative once you add the one number it does not print.

**The noise floor is 0.01.** The signal is `sin(phase) + 0.1*randn`, so each sample carries independent noise of variance $0.1^2$. Noise is by definition unpredictable, so **no** model — linear, recurrent, or otherwise — can achieve a one-step MSE below 0.01. Measured against that ceiling:

| model | test MSE | × the floor | excess over floor |
|---|---|---|---|
| persistence | 0.03177 | 3.18× | 0.0218 |
| linear AR | 0.01421 | 1.42× | 0.0042 |
| LSTM | 0.01776 | 1.78× | 0.0078 |

Now the result reads differently. Both fitted models are close to optimal, and the *entire* contest was over the 0.0042 of predictable structure that AR captured and the LSTM did not quite. The LSTM did not fail; it solved a nearly-solved problem slightly less cleanly, with a hundred times the parameters.

**Why AR was always going to win here.** One-step-ahead prediction of a smooth oscillation is a nearly linear operation — a locally sinusoidal signal is well described by a low-order linear recurrence, which is exactly what a 40-tap AR model represents. The hypothesis class *matches the task*. The LSTM's nonlinearity buys nothing because there is no nonlinearity to capture, and it costs the optimisation difficulty of finding a good solution in a much larger space. More epochs would narrow the gap and could not meaningfully surpass 0.01.

**So the lesson is not "linear models win."** It is *match the model class to the task, and measure against a floor and a baseline before believing anything*. Push the task past linearity and the ranking flips: predict 20 steps ahead instead of 1, deepen the frequency wobble, or raise the noise. Iterated multi-step prediction compounds error through the model's own dynamics, which is where a learned nonlinear state-space model earns its parameters. Running that variant live is the fastest way to see that neither model is better in general.

And persistence deserves its row. Zero parameters, no fitting, 3.2× the floor — a forecaster that cannot beat *that* has demonstrated nothing at all. Publishing a neural forecaster without both baselines is how the literature accumulates results that quietly do not replicate.

In [8]:
plt.figure(figsize=(9, 2.8))
plt.plot(truth[:300], "k", linewidth=1, label="truth")
plt.plot(pred_ar[:300], label="linear AR", alpha=0.8)
plt.plot(pred_lstm[:300], label="LSTM", alpha=0.8)
plt.legend(); plt.title("One-step-ahead forecasts on unseen data")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1845167/21433818.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Read that table carefully: **the linear AR model wins** — and that is the most important lesson in this workshop. One-step-ahead prediction of a smooth oscillation is a *nearly linear* task, and 40 least-squares coefficients beat 4k+ trained parameters. Deep models earn their keep only when the task outgrows linearity: try deepening the frequency wobble, raising the noise, or predicting 20 steps ahead instead of 1 — and watch the ranking flip. Never publish a neural forecaster without this bake-off.

## 4. Conclusion

An RNN is a learned nonlinear state-space model: unroll it to train it, gate it to remember, clip it to keep it stable. And always race it against persistence and a linear model before celebrating.

---
## Where next

- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — replace recurrence with attention: all time steps talk directly, no vanishing highway needed.
- [Adaptive Filtering: Kalman](./Intro_AdFilt_KF.ipynb) — when you *know* the dynamics, the hand-derived gain is still king.
- [Scaling Neural Networks](../Intro_Mach_Learn/README.md#workshop-3--scaling-neural-networks-available) — what happens when models like these grow 1000×.